In [1]:
import os
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

# Use tf_keras (TensorFlow Keras) to avoid keras==3.x protobuf/compat issues in this environment.
import tf_keras as keras
from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.models import Sequential
from tf_keras.layers import (
    Dense,
    Conv2D,
    Dropout,
    MaxPooling2D,
    Activation,
    BatchNormalization,
    GlobalAveragePooling2D,
)
from tf_keras.callbacks import ReduceLROnPlateau
from tf_keras.regularizers import l2

# Paths (match the provided Kaggle filesystem listing)
BASE_PATH = "/kaggle/input/aerial-cactus-identification"
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")
SAMPLE_SUB = os.path.join(BASE_PATH, "sample_submission.csv")




2026-01-10 23:39:51.032360: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768088391.046097      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768088391.050330      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-10 23:39:51.068114: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
def load_data(dataframe=None, batch_size=32, mode="categorical", seed=42):
    if dataframe is None:
        dataframe = pd.read_csv(TRAIN_CSV)

    # Keras generator expects string labels for categorical mode
    dataframe = dataframe.copy()
    dataframe["has_cactus"] = dataframe["has_cactus"].astype(str)

    gen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        validation_split=0.1,
        horizontal_flip=True,
        vertical_flip=True,
    )

    trainGen = gen.flow_from_dataframe(
        dataframe=dataframe,
        directory=TRAIN_DIR,
        x_col="id",
        y_col="has_cactus",
        target_size=(32, 32),
        class_mode=mode,
        batch_size=batch_size,
        shuffle=True,
        subset="training",
        seed=seed,
    )

    valGen = gen.flow_from_dataframe(
        dataframe=dataframe,
        directory=TRAIN_DIR,
        x_col="id",
        y_col="has_cactus",
        target_size=(32, 32),
        class_mode=mode,
        batch_size=batch_size,
        shuffle=False,
        subset="validation",
        seed=seed,
    )

    return trainGen, valGen


trainGen, valGen = load_data(batch_size=32)




Found 12758 validated image filenames belonging to 2 classes.
Found 1417 validated image filenames belonging to 2 classes.


In [3]:
def train_model():
    model = Sequential()
    model.add(
        Conv2D(
            32,
            (3, 3),
            padding="same",
            kernel_regularizer=l2(1e-4),
            input_shape=(32, 32, 3),
        )
    )
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(32, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(64, (3, 3), kernel_regularizer=l2(1e-4), padding="same"))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(64, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(128, (3, 3), kernel_regularizer=l2(1e-4), padding="same"))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(Conv2D(128, (3, 3), kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(BatchNormalization())

    model.add(GlobalAveragePooling2D())
    model.add(Dense(2))
    model.add(BatchNormalization())
    model.add(Activation("softmax"))

    return model




In [4]:
model = train_model()

# Keras 2.x uses learning_rate (lr is deprecated/ignored in some configs)
opt = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

cbs = [
    ReduceLROnPlateau(monitor="loss", factor=0.5, patience=1, min_lr=1e-5, verbose=1)
]

# Use automatic steps_per_epoch/validation_steps from generators to avoid overshooting/long runs.
model.fit(
    trainGen,
    epochs=3,
    validation_data=valGen,
    shuffle=True,
    callbacks=cbs,
    verbose=1,
)



2026-01-10 23:39:58.141648: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768088398.142117      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


Epoch 1/3


E0000 00:00:1768088399.701998      11 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1768088399.879227      65 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1768088401.035371      64 service.cc:148] XLA service 0x7f45b3089f70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768088401.035409      64 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-10 23:40:01.041265: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768088401.104327      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


399/399 [==============================] - 8s 9ms/step - loss: 0.3800 - accuracy: 0.8949 - val_loss: 1.4784 - val_accuracy: 0.2971 - lr: 1.0000e-04
Epoch 2/3
399/399 [==============================] - 3s 8ms/step - loss: 0.3102 - accuracy: 0.9439 - val_loss: 0.3535 - val_accuracy: 0.9160 - lr: 1.0000e-04
Epoch 3/3
399/399 [==============================] - 3s 8ms/step - loss: 0.2744 - accuracy: 0.9598 - val_loss: 0.4580 - val_accuracy: 0.8582 - lr: 1.0000e-04


In [5]:
test_set = pd.read_csv(SAMPLE_SUB)

pred = np.empty((test_set.shape[0],), dtype=np.float32)
for n in tqdm(range(test_set.shape[0])):
    img_path = os.path.join(TEST_DIR, test_set.id.iloc[n])
    data = np.array(Image.open(img_path))
    data = data.astype(np.float32) / 255.0
    # probability for class index 1 corresponds to '1' label given sorted class indices ['0','1']
    pred[n] = model.predict(data.reshape((1, 32, 32, 3)), verbose=0)[0][1]

submission = pd.DataFrame({"id": test_set["id"], "has_cactus": pred})
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", submission.shape)

100%|██████████| 3325/3325 [01:43<00:00, 32.02it/s]

Wrote submission.csv with shape: (3325, 2)
